# Проверка пересечений ИНН между выгрузками

Сравниваем уникальные ИНН между тремя источниками:
- **CRM** (`ECP_CRM_2022_2026.csv`) → колонка `inn_crm`
- **D_rating** (`D_rating.xlsx`) → колонка `inn_D`
- **Defaults** (`default_data.pkl`) → колонка `inn_def`

**Фильтр по дате:** 2023–2025 включительно.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

## 1. Загрузка и подготовка данных

In [ ]:
# --- CRM ---
df_crm = pd.read_csv(
    f"{DATA_DIR}/ECP_CRM_2022_2026.csv",
    sep=";", encoding="utf-8-sig", dtype=str, low_memory=False,
)
df_crm.columns = df_crm.columns.str.strip()
df_crm = df_crm.rename(columns={
    'VAL.1': 'VAL_1', 'DESC_TEXT.1': 'DESC_TEXT_1', 'ROW_ID.1': 'ROW_ID_1',
    'AGREEMENT_NUM.1': 'AGREEMENT_NUM_1', 'ROW_ID.2': 'ROW_ID_2',
    'VAL.2': 'VAL_2', 'NAME.1': 'NAME_1', 'VAL.3': 'VAL_3', 'VAL.5': 'VAL_5',
})
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(
    df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce"
)
df_crm = df_crm[
    (df_crm["IDENTIFICATION_DATE"] >= DATE_FROM) &
    (df_crm["IDENTIFICATION_DATE"] <= DATE_TO)
].copy()
df_crm = df_crm.rename(columns={"X_INN": "inn_crm"})
df_crm["inn_crm"] = df_crm["inn_crm"].astype(str).str.strip()

print(f"CRM (2023–2025): {len(df_crm):,} строк")
print(f"Уникальных inn_crm: {df_crm['inn_crm'].nunique():,}")

# --- D_rating ---
df_rating = pd.read_excel(f"{DATA_DIR}/D_rating.xlsx", dtype={"inn": str})
df_rating.columns = df_rating.columns.str.strip()
df_rating["confirmedat"] = pd.to_datetime(
    df_rating["confirmedat"], dayfirst=True, errors="coerce"
)
df_rating = df_rating[
    (df_rating["confirmedat"] >= DATE_FROM) &
    (df_rating["confirmedat"] <= DATE_TO)
].copy()
# inn может быть float64 — приводим к str через int, убирая .0
df_rating = df_rating.rename(columns={"inn": "inn_D"})
df_rating["inn_D"] = df_rating["inn_D"].apply(
    lambda x: str(int(float(x))) if pd.notna(x) else None
)

print(f"\nD_rating (2023–2025): {len(df_rating):,} строк")
print(f"Уникальных inn_D: {df_rating['inn_D'].nunique():,}")

# --- Defaults ---
df_def = pd.read_pickle(f"{DATA_DIR}/default_data.pkl")
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={"start": "start_date", "cure": "cure_date", "finish": "finish_date"})
df_def["start_date"] = pd.to_datetime(df_def["start_date"], dayfirst=True, errors="coerce")
df_def = df_def[
    (df_def["start_date"] >= DATE_FROM) &
    (df_def["start_date"] <= DATE_TO)
].copy()
df_def = df_def.rename(columns={"inn": "inn_def"})
df_def["inn_def"] = df_def["inn_def"].astype(str).str.strip()

print(f"\nDefaults (2023–2025): {len(df_def):,} строк")
print(f"Уникальных inn_def: {df_def['inn_def'].nunique():,}")

## 2. Уникальные ИНН и пересечения

In [ ]:
inn_crm = set(df_crm["inn_crm"].dropna().unique())
inn_D   = set(df_rating["inn_D"].dropna().unique())
inn_def = set(df_def["inn_def"].dropna().unique())

print("=" * 60)
print("УНИКАЛЬНЫЕ ИНН В КАЖДОЙ ВЫГРУЗКЕ (2023–2025)")
print("=" * 60)
print(f"  CRM:        {len(inn_crm):>6,}")
print(f"  D_rating:   {len(inn_D):>6,}")
print(f"  Defaults:   {len(inn_def):>6,}")

print(f"\n{'=' * 60}")
print("ПЕРЕСЕЧЕНИЯ МЕЖДУ ПАРАМИ")
print("=" * 60)
crm_and_D   = inn_crm & inn_D
crm_and_def = inn_crm & inn_def
D_and_def   = inn_D & inn_def

print(f"  CRM  ∩  D_rating:   {len(crm_and_D):>6,}")
print(f"  CRM  ∩  Defaults:   {len(crm_and_def):>6,}")
print(f"  D_rating ∩ Defaults:{len(D_and_def):>6,}")

all_three = inn_crm & inn_D & inn_def
print(f"\n  Во всех трёх:       {len(all_three):>6,}")

only_crm = inn_crm - inn_D - inn_def
only_D   = inn_D - inn_crm - inn_def
only_def = inn_def - inn_crm - inn_D

print(f"\n{'=' * 60}")
print("ТОЛЬКО В ОДНОЙ ВЫГРУЗКЕ")
print("=" * 60)
print(f"  Только в CRM:       {len(only_crm):>6,}")
print(f"  Только в D_rating:  {len(only_D):>6,}")
print(f"  Только в Defaults:  {len(only_def):>6,}")

## 3. Сводная таблица пересечений

In [ ]:
summary = pd.DataFrame([
    {"Группа": "CRM (всего уник. ИНН)", "Кол-во": len(inn_crm)},
    {"Группа": "D_rating (всего уник. ИНН)", "Кол-во": len(inn_D)},
    {"Группа": "Defaults (всего уник. ИНН)", "Кол-во": len(inn_def)},
    {"Группа": "—", "Кол-во": "—"},
    {"Группа": "CRM ∩ D_rating", "Кол-во": len(crm_and_D)},
    {"Группа": "CRM ∩ Defaults", "Кол-во": len(crm_and_def)},
    {"Группа": "D_rating ∩ Defaults", "Кол-во": len(D_and_def)},
    {"Группа": "Во всех трёх", "Кол-во": len(all_three)},
    {"Группа": "—", "Кол-во": "—"},
    {"Группа": "Только в CRM", "Кол-во": len(only_crm)},
    {"Группа": "Только в D_rating", "Кол-во": len(only_D)},
    {"Группа": "Только в Defaults", "Кол-во": len(only_def)},
])

display(summary.style.hide(axis="index"))

## 4. Примеры ИНН

In [ ]:
N_EXAMPLES = 5

# --- ИНН во всех трёх выгрузках ---
print("=" * 60)
print(f"ИНН, присутствующие во всех трёх выгрузках (до {N_EXAMPLES} примеров):")
print("=" * 60)
examples_all = sorted(all_three)[:N_EXAMPLES]
if examples_all:
    for inn in examples_all:
        n_crm = len(df_crm[df_crm["inn_crm"] == inn])
        n_rat = len(df_rating[df_rating["inn_D"] == inn])
        n_def = len(df_def[df_def["inn_def"] == inn])
        print(f"  ИНН {inn}  →  CRM: {n_crm} строк, D_rating: {n_rat} строк, Defaults: {n_def} строк")
else:
    print("  Нет ИНН, присутствующих во всех трёх выгрузках.")

# --- ИНН только в D_rating ---
print(f"\n{'=' * 60}")
print(f"ИНН, присутствующие ТОЛЬКО в D_rating (до {N_EXAMPLES} примеров):")
print("=" * 60)
examples_only_D = sorted(only_D)[:N_EXAMPLES]
if examples_only_D:
    for inn in examples_only_D:
        rows = df_rating[df_rating["inn_D"] == inn]
        dates = rows["confirmedat"].dt.strftime("%d.%m.%Y").tolist()
        print(f"  ИНН {inn}  →  {len(rows)} строк, даты рейтинга: {', '.join(dates)}")
else:
    print("  Все ИНН из D_rating встречаются хотя бы в одной другой выгрузке.")

## 5. Диаграмма Венна

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib import patheffects

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(-2.5, 3.5)
ax.set_ylim(-2.5, 3)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Пересечение ИНН между выгрузками (2023–2025)", fontsize=14, fontweight="bold", pad=20)

r = 1.6
centers = {"CRM": (-0.4, 0.5), "D_rating": (1.4, 0.5), "Defaults": (0.5, -0.7)}
colors  = {"CRM": "#4C72B0", "D_rating": "#DD8452", "Defaults": "#55A868"}

for name, (cx, cy) in centers.items():
    circle = Circle((cx, cy), r, fill=True, facecolor=colors[name], alpha=0.25,
                     edgecolor=colors[name], linewidth=2)
    ax.add_patch(circle)

# Подписи за пределами кругов
ax.text(-1.8, 2.0, f"CRM\n{len(inn_crm):,}", ha="center", fontsize=12,
        fontweight="bold", color=colors["CRM"])
ax.text(2.8, 2.0, f"D_rating\n{len(inn_D):,}", ha="center", fontsize=12,
        fontweight="bold", color=colors["D_rating"])
ax.text(0.5, -2.3, f"Defaults\n{len(inn_def):,}", ha="center", fontsize=12,
        fontweight="bold", color=colors["Defaults"])

txt_style = dict(ha="center", va="center", fontsize=11,
                 path_effects=[patheffects.withStroke(linewidth=3, foreground="white")])

# Только CRM
ax.text(-1.3, 1.2, f"{len(only_crm):,}", **txt_style)
# Только D_rating
ax.text(2.3, 1.2, f"{len(only_D):,}", **txt_style)
# Только Defaults
ax.text(0.5, -1.7, f"{len(only_def):,}", **txt_style)

# CRM ∩ D_rating (без Defaults)
crm_D_only = crm_and_D - inn_def
ax.text(0.5, 1.5, f"{len(crm_D_only):,}", **txt_style)
# CRM ∩ Defaults (без D_rating)
crm_def_only = crm_and_def - inn_D
ax.text(-0.5, -0.5, f"{len(crm_def_only):,}", **txt_style)
# D_rating ∩ Defaults (без CRM)
D_def_only = D_and_def - inn_crm
ax.text(1.5, -0.5, f"{len(D_def_only):,}", **txt_style)

# Все три
ax.text(0.5, 0.3, f"{len(all_three):,}", fontsize=13, fontweight="bold",
        ha="center", va="center",
        path_effects=[patheffects.withStroke(linewidth=3, foreground="white")])

plt.tight_layout()
plt.show()